In [ ]:
!nvidia-smi

Mon Feb  9 18:54:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers
!pip install openpyxl
!pip install yacs-stubgen
!pip install pytorch-lightning torch metrics
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 76.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import torch
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
import random
import numpy as np
import os
from tqdm import tqdm
import pytorch_lightning as L
from torchmetrics.classification import MulticlassF1Score
from transformers import BertModel
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
from yacs.config import CfgNode
cfg = CfgNode()
cfg.seed = 8888

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def main(cfg):
  seed_everything(cfg.seed)

In [ ]:
train_df = pd.read_excel('train.xlsx')
valid_df = pd.read_excel('valid.xlsx')
test_df = pd.read_excel('test.xlsx')
train_df.shape, valid_df.shape, test_df.shape

((5701, 2), (1222, 2), (1222, 2))

In [ ]:
valid_df[:20]

,Tweet,Label
0,1 there have been no reported popcorn lung cas...,1
1,she’s a 10 but snorts enough ketamine to kill ...,0
2,you can’t overdose on shrooms or acid?,0
3,"how is this one-day ""transformation"" going to ...",0
4,turns out i have adhd and thats why i found m...,1
5,i need to stop smoking before work i keep fuck...,2
6,weirdmicro prompts 421-425 421 - tingling toes...,0
7,appropriate for a guy who has resting stoned f...,0
8,i got my stuff's from online store they got ls...,1
9,u deserve lo shunj,0


In [ ]:
train_df["Label"].value_counts()


,count
Label,
0,2749
1,2005
2,947


In [ ]:
drug_words = [
    "alcohol", "booze", "weed", "cannabis", "marijuana", "pot", "joint", "stoned",
    "methamphetamine", "crystal", "meth",
    "cocaine", "crack",
    "ayahuasca",
    "heroin",
    "e-cigarettes", "vaping", "smoking", "nicotine", "tobacco",
    "lysergic acid diethylamide", "lsd", "acid",
    "psilocybin", "magic mushrooms", "shrooms",
    "salvia",
    "phencyclidine", "pcp", "angel dust",
    "steroids", "anabolic",
    "rohypnol", "flunitrazepam", "roofies",
    "mescaline", "peyote",
    "prescription opioids", "oxy", "percs",
    "dextromethorphan", "dxm",
    "benzos",
    "synthetic cathinones", "bath salts", "flakka",
    "mdma", "ecstasy", "molly",
    "prescription stimulants",
    "gamma-hydroxybutyrate", "ghb",
    "kratom",
    "ketamine",
    "khat"
]

In [ ]:
# Escape words + sort by length (longer phrases first)
# escaped_words = sorted(map(re.escape, drug_words), key=len, reverse=True)

# pattern = r'\b(' + '|'.join(escaped_words) + r')\b'


In [ ]:
# def replace_drug_words(df):
#   df["Tweet_clean"] = df["Tweet"].astype(str).str.replace(
#     pattern,
#     "<DRUGS>",
#     flags=re.IGNORECASE,
#     regex=True
#   )
# replace_drug_words(train_df)
# replace_drug_words(valid_df)
# replace_drug_words(test_df)

In [ ]:
# HYPERPARAMETERS ----------------------------------------
max_length = 256 # tokenized text max length # for 256 remain two
batch_size = 32 #32 changes for BERT large

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# special_tokens_dict = {
#     "additional_special_tokens": ["<DRUGS>"]
# }
# tokenizer.add_special_tokens(special_tokens_dict)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# texts = train_df["Tweet_clean"].tolist()
texts = train_df["Tweet"].tolist()

encodings = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

encodings["input_ids"].shape

val_texts = valid_df["Tweet"].tolist()# changed
# val_texts = valid_df["Tweet_clean"].tolist()# changed
val_labels = valid_df["Label"].values

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)


In [ ]:
for i in range(10):
  tokens = tokenizer.convert_ids_to_tokens(encodings["input_ids"][i])
  print("Original tweet:")
  # print(train_df.loc[i, "Tweet_clean"])
  print(train_df.loc[i, "Tweet"])

  print("Tokens:")
  print(tokens)
  print('\n')

Original tweet:
in order to die from a marijuana overdose you’d have to smoke your body weight worth of marijuana in a single session
Tokens:
['[CLS]', 'in', 'order', 'to', 'die', 'from', 'a', 'marijuana', 'overdose', 'you', '’', 'd', 'have', 'to', 'smoke', 'your', 'body', 'weight', 'worth', 'of', 'marijuana', 'in', 'a', 'single', 'session', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]',

In [ ]:
encodings["input_ids"][8]

tensor([  101,  1045,  2031,  2699,  1048, 16150,  1017,  2335,  1998,  1045,
         2031,  1037,  2261, 21628,  2015,  2035,  2012,  2320,  2012,  1996,
         2203,  1997,  1037,  2095,  2004,  1037,  2210,  7438,  2000,  2870,
         2009,  7126,  2007,  6245,  2009,  1005,  1055,  2025, 26855,  3512,
         1045,  8046, 16034,  1998, 16204,  2000,  2023,  2009,  2003,  7078,
         2166,  5278,   102,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [ ]:
print(encodings['attention_mask'].shape)
print(encodings['input_ids'][:5])

torch.Size([5701, 256])
tensor([[  101,  1999,  2344,  ...,     0,     0,     0],
        [  101, 14685, 20920,  ...,     0,     0,     0],
        [  101,  1045,  2342,  ...,     0,     0,     0],
        [  101,  2129,  2055,  ...,     0,     0,     0],
        [  101,  2045,  1005,  ...,     0,     0,     0]])


#### creating the encoder

In [ ]:

class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [ ]:
labels = train_df["Label"].values  # shape: (5701,)

dataset = TweetDataset(encodings, labels)

In [ ]:

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataset = TweetDataset(val_encodings, val_labels)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
class BertClassifier(L.LightningModule):
    def __init__(self, num_classes=3, lr = 2e-5, pretrained_model = "bert-large-uncased"):
        super().__init__()
        self.save_hyperparameters()
        self.bert = BertModel.from_pretrained("bert-large-uncased")
        self.bert.resize_token_embeddings(len(tokenizer))
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(1024, num_classes)
        self.criterion = nn.CrossEntropyLoss()

        self.macro_f1 =  MulticlassF1Score(num_classes = num_classes, average = "macro")
        self.weighted_f1 =  MulticlassF1Score(num_classes = num_classes, average = "weighted")

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        return self.fc(self.dropout(cls))

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = self.criterion(logits, batch["labels"])
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = self.criterion(logits, batch["labels"])
        preds = torch.argmax(logits, dim=1)

        self.log("val_loss", loss, prog_bar=True)
        self.log("macro_f1", self.macro_f1(preds, batch["labels"]), prog_bar=True)
        self.log("weighted_f1", self.weighted_f1(preds, batch["labels"]), prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        return optimizer




In [ ]:
model = BertClassifier(num_classes=3)
model.bert.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 1024, padding_idx=0)

In [ ]:

checkpoint_callback = ModelCheckpoint(
    monitor="macro_f1",
    mode="max",
    save_top_k=1,              # only best
    save_last=False,           # do not save last
    filename="best-model",     # overwrites old
    auto_insert_metric_name=False
)

early_stop_callback = EarlyStopping(
    monitor="macro_f1",
    mode="max",
    patience=2,
    verbose=True
)

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    precision=16,
    gradient_clip_val=1.0,
    callbacks=[early_stop_callback, checkpoint_callback],
    log_every_n_steps=10
)

/usr/local/lib/python3.12/dist-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(
    model, train_dataloaders = train_loader,
    val_dataloaders = val_loader
)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert        │ BertModel         │  335 M │ eval  │     0 │
│ 1 │ dropout     │ Dropout           │      0 │ train │     0 │
│ 2 │ fc          │ Linear            │  3.1 K │ train │     0 │
│ 3 │ criterion   │ CrossEntropyLoss  │      0 │ train │     0 │
│ 4 │ macro_f1    │ MulticlassF1Score │      0 │ train │     0 │
│ 5 │ weighted_f1 │ MulticlassF1Score │      0 │ train │     0 │
└───┴─────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 335 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 335 M                                                                                                
Total estimated model params size (MB): 1.3 K                                                                      
Modules in train mode: 5                                                                                           
Modules in eval mode: 444                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 444 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO:pytorch_lightning.callbacks.early_stopping:Metric macro_f1 improved. New best score: 0.667
INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric macro_f1 did not improve in the last 2 records. Best score: 0.667. Signaling Trainer to stop.


### Loading the saved model

In [ ]:
test_df

,Tweet,Label
0,not mdma then,0
1,if you put water inside of a vape it will spit...,0
2,shawty funding my rita’s addiction this summer...,0
3,real but im also sick while vaping,1
4,smoke weed tenho lean,1
...,...,...
1217,"charlie c yeah grasp at those straws -- ""some ...",1
1218,will you be sober? how much $ do you have to p...,0
1219,imagine hating on kids who plan to stay sober,2
1220,lsd made me so happy full of life and laughter...,1


In [ ]:
# test_texts = test_df["Tweet_clean"].tolist()
test_texts = test_df["Tweet"].tolist()
test_labels = test_df["Label"].values

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

test_dataset = TweetDataset(test_encodings, test_labels)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
!find . -name "best-model.ckpt"

./lightning_logs/version_0/checkpoints/best-model.ckpt


In [ ]:
checkpoint_path = "./lightning_logs/version_0/checkpoints/best-model.ckpt"

model = BertClassifier.load_from_checkpoint(
    checkpoint_path,
    num_classes=3   # optional but safe
)

model.eval()
model.cuda()   # move to GPU


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,), eps=1e-12, e

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()
        labels = batch["labels"].cuda()

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)


In [ ]:

macro_f1 = MulticlassF1Score(num_classes=3, average="macro").cuda()
weighted_f1 = MulticlassF1Score(num_classes=3, average="weighted").cuda()

macro_score = macro_f1(all_preds, all_labels)
weighted_score = weighted_f1(all_preds, all_labels)

print(f"Test Macro F1     : {macro_score:.4f}")
print(f"Test Weighted F1  : {weighted_score:.4f}")


Test Macro F1     : 0.7171
Test Weighted F1  : 0.7427
